## Drone Localisation Simulation
This project is setup to test the simulation of a microphone picking up the sounds of a drone for the purposes of localisation.



In [ ]:
!pip install pyroomacoustics numpy scipy==1.11.1 matplotlib acoular graphviz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyroomacoustics as pra
from numpy import hamming
from scipy.io import wavfile
from IPython.display import Audio, display
from math import floor
import acoular
import os
import h5py
import tables
import time

## Building the Space

This is where we specify the dimensions of our simulation environment.

In [ ]:
def append_grid_coordinates(
    start_x: float,
    start_y: float,
    grid_side_length: int,
    mic_spacing: float,
    prev_all_x_coords: np.ndarray | None = None,
    prev_all_y_coords: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates a 2D grid of coordinates and appends it to existing coordinate arrays. This allows easy generation of multiple microphone arrays.

    Args:
        start_x: The starting x-coordinate for the new grid.
        start_y: The starting y-coordinate for the new grid.
        grid_side_length: The number of points along one side of the grid.
        mic_spacing: The spacing between points in the grid.
        prev_all_x_coords: An optional numpy array of existing x-coordinates.
                           If None, a new array is created.
        prev_all_y_coords: An optional numpy array of existing y-coordinates.
                           If None, a new array is created.

    Returns:
        A tuple containing two numpy arrays:
        - The new, concatenated array of all x-coordinates.
        - The new, concatenated array of all y-coordinates.
    """
    # 1. Generate the 1D coordinate vectors for the new grid.
    # np.linspace creates an array of evenly spaced numbers over a specified interval.
    x_coords_1d = np.linspace(start_x, start_x + (grid_side_length - 1) * mic_spacing, grid_side_length)
    y_coords_1d = np.linspace(start_y, start_y + (grid_side_length - 1) * mic_spacing, grid_side_length)

    # 2. Use np.meshgrid to create the 2D grid of (X, Y) pairs for the new grid.
    X_grid_new, Y_grid_new = np.meshgrid(x_coords_1d, y_coords_1d)

    # 3. Flatten the new grid into 1D arrays of its coordinates.
    x_coords_to_add = X_grid_new.flatten()
    y_coords_to_add = Y_grid_new.flatten()

    # 4. Handle the concatenation logic.
    # If there are no previous coordinates, the new coordinates are just the ones we just generated.
    if prev_all_x_coords is None or prev_all_y_coords is None:
        all_x_coords = x_coords_to_add
        all_y_coords = y_coords_to_add
    else:
        # If previous coordinates exist, append the new ones.
        all_x_coords = np.concatenate([prev_all_x_coords, x_coords_to_add])
        all_y_coords = np.concatenate([prev_all_y_coords, y_coords_to_add])

    return all_x_coords, all_y_coords

In [ ]:
# Simulation space set up- square box
corner_locs = 150.0

corners = np.array(
    [
        [0.0,0.0],
        [corner_locs,0.0],
        [corner_locs,corner_locs],
        [0.0,corner_locs]
    ]
).T
h = 9.5

# Microphone array setup
GRID_SIDE_LENGTH = 2
MIC_SPACING = 0.5
fixed_z = 1.2

# Source Creation
source = np.array([[120], [100], [4.0]])
# source = np.array([[50], [0], [4.0]])
source_noise = np.array([[114], [40], [2.0]])

# First Microphone Array
all_x_coords, all_y_coords = append_grid_coordinates(
    start_x=5.0,
    start_y=5.0,
    grid_side_length=GRID_SIDE_LENGTH,
    mic_spacing=MIC_SPACING,
    prev_all_x_coords=None,
    prev_all_y_coords=None
)

# Second Microphone Array
all_x_coords, all_y_coords = append_grid_coordinates(
    start_x=120.0,
    start_y=5.0,
    grid_side_length=GRID_SIDE_LENGTH,
    mic_spacing=MIC_SPACING,
    prev_all_x_coords=all_x_coords,
    prev_all_y_coords=all_y_coords
)


# Third Microphone Array
all_x_coords, all_y_coords = append_grid_coordinates(
    start_x=5.0,
    start_y=120.0,
    grid_side_length=GRID_SIDE_LENGTH,
    mic_spacing=MIC_SPACING,
    prev_all_x_coords=all_x_coords,
    prev_all_y_coords=all_y_coords
)




# Create the Z coordinates (all fixed at 'fixed_z')
all_z_coords = np.full(all_x_coords.shape[0], fixed_z)

# Preferred and most readable way:
mic_locs = np.array([all_x_coords, all_y_coords, all_z_coords])


## Displaying the Space


In [ ]:
# Display the Space in 2D
fs, s = wavfile.read("drone_audio.wav") # Get actual sample rate from clip
room = pra.Room.from_corners(corners)
room.add_source(source[:2])
room.add_source(source_noise[:2])
room.add_microphone_array(pra.MicrophoneArray(mic_locs[:2,:], fs=fs))

fig, ax = room.plot(img_order=2, mic_marker_size = 40)
ax.set_xlim([-1, 1 + corner_locs])
ax.set_ylim([-1, 1 + corner_locs])


In [ ]:
# Set wall materials for open field
wall_material = pra.Material(energy_absorption=1.0, scattering=0.0)
ceiling_material = pra.Material(energy_absorption=1.0, scattering=0.0)
floor_material = pra.Material(energy_absorption=1.0, scattering=0.0)

# Redefine room for 3D raytracing simulation.
room = pra.Room.from_corners(corners, fs=fs, max_order=5, materials=wall_material, ray_tracing=True, air_absorption=True)
room.extrude(h, materials=ceiling_material)
room.set_ray_tracing(receiver_radius=0.1, n_rays=10000, energy_thres=1e-7)
room.add_source(source)
room.add_microphone_array(pra.MicrophoneArray(mic_locs, fs=fs))

# Compute image sources
room.image_source_model()
room.plot_rir()
fig = plt.gcf()
fig.set_size_inches(20, 10)

t60 = pra.experimental.measure_rt60(room.rir[0][0], fs=room.fs, plot=False)
print(f"The RT60 is {t60 * 1000:.0f} ms")

In [ ]:
# The desired reverberation time and dimensions of the room
rt60_tgt = 1 # seconds
room_dim = [corner_locs,corner_locs,h] # meters

# import a mono wavfile as the source signal
# the sampling frequency should match that of the room
fs, audio = wavfile.read("drone_audio.wav")
fs, audio_noise = wavfile.read("noise.wav")

# noise
noise = True
noise_attenuation = 0.0000005
audio_noise = audio_noise * noise_attenuation

if audio.ndim > 1:
    audio = np.mean(audio, axis=1)

if audio_noise.ndim > 1:
    audio_noise = np.mean(audio_noise, axis=1)

# We invert Sabine's formula to obtain the parameters for the ISM simulator
e_absorption, max_order = pra.inverse_sabine(rt60_tgt, room_dim)


# Create the room
room = pra.ShoeBox(
    room_dim, fs=fs, materials=pra.Material(1.0), max_order=max_order
)


# place the source in the room
room.add_source(source, signal=audio, delay=0.5)
if noise:
  room.add_source(source_noise, signal=audio_noise, delay=0.25)


# finally place the array in the room
room.add_microphone_array(mic_locs)

# Run the simulation (this will also build the RIR automatically)
room.simulate()

"""
room.mic_array.to_wav(
    f"/content/mic_output.wav",
    norm=True,
    bitdepth=np.int16,
)
"""

# measure the reverberation time
rt60 = room.measure_rt60()
print("The desired RT60 was {}".format(rt60_tgt))
print("The measured RT60 is {}".format(rt60[1, 0]))

# Create a plot
plt.figure()

# plot one of the RIR. both can also be plotted using room.plot_rir()
rir_1_0 = room.rir[1][0]

plt.plot(np.arange(len(rir_1_0)) / room.fs, rir_1_0)
plt.title("The RIR from source 0 to mic 1")
plt.xlabel("Time [s]")
plt.show()

# plot signal at microphone 1

plt.plot(audio)
plt.title("Raw signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(audio_noise)
plt.title("Noise")
plt.xlabel("Time [s]")
plt.show()


plt.plot(room.mic_array.signals[0, :])
plt.title("Microphone 1 Signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(room.mic_array.signals[1, :])
plt.title("Microphone 2 Signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(room.mic_array.signals[2, :])
plt.title("Microphone 3 Signal")
plt.xlabel("Time [s]")
plt.show()



audio_data1 = room.mic_array.signals[0, :]
audio_data2 = room.mic_array.signals[1, :]
print("Unaltered Drone Sound: Normalized")
display(Audio(data=audio, rate=fs))
print("Unaltered Noise Sound: Normalized")
display(Audio(data=audio_noise, rate=fs))
print("Microphone 1: Normalized")
display(Audio(data=audio_data1, rate=fs))
print("Microphone 2: Normalized")
display(Audio(data=audio_data2, rate=fs))

## Method to delete h5 file

Due to the unique properties h5 files you will have difficulties editing them without special considerations. The easier method is to recreate this h5 file every time. If you have not run this code you do not need to run the next code block


In [ ]:
files = ["sim_signals.h5", "three_sources.h5"]

for file_name in files:
    if os.path.exists(file_name):
        print(f"File '{file_name}' exists. Attempting to close any open HDF5/PyTables handles...")

        try:
            # This is the most effective way to close HDF5 files opened by this process
            # (via h5py or PyTables).
            tables.file._open_files.close_all()
            print("Successfully attempted to close all open HDF5/PyTables files within this process.")
        except ImportError:
            print("PyTables is not installed, skipping tables.file._open_files.close_all().")
            print("If you frequently encounter file locking issues, consider 'pip install tables'.")
        except Exception as e:
            print(f"Error while trying to close HDF5/PyTables files: {e}")

        # Give a brief moment for the OS to release the handle
        time.sleep(0.1)

        # Now, attempt to delete the file
        try:
            os.remove(file_name)
            print(f"File '{file_name}' deleted successfully.")
        except OSError as e:
            print(f"Error deleting file '{file_name}': {e}")
            print("This often indicates another *external* process (not this Python script) is holding the file open.")
            print("Please ensure no other applications (like HDFView, another Python script, etc.) are using 'sim_signals.h5'.")
    else:
        print(f"File '{file_name}' does not exist.")

In [ ]:
import scipy.fft as fft

N = len(audio)
T = 1.0 / fs
yf = fft.fft(audio)
xf = fft.fftfreq(N, T)[:N//2]

plt.figure(figsize=(10, 4))
plt.plot(xf, 2.0/N * np.abs(yf[0:N//2]))
plt.title("FFT of Drone Audio Signal")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.xlim(0, 16000) # Focus on relevant range
plt.grid(True)
plt.show()

## Localisation using Acoular

In [ ]:

# Use mic positions directly (shape must be (3, N))
mic_positions = room.mic_array.R  # shape: (3, num_mics)

mg = acoular.MicGeom()
mg.pos_total = mic_positions  # no need for .T, Acoular expects (3, N)

print("MicGeom initialized with", mg.num_mics, "mics")


In [ ]:
import h5py
import os

mic_signals = room.mic_array.signals  # Or however you've stored it
sample_freq = fs  # or whatever you've used

with h5py.File('sim_signals.h5', 'w') as f:
    f.create_dataset('time_data', data=mic_signals.T)  # Transpose to shape (samples, channels)
    f.attrs['sample_freq'] = sample_freq


In [ ]:
with h5py.File('sim_signals.h5', 'r') as f:
    data = f['time_data']
    print("Shape of time_data:", data.shape)

In [ ]:
import h5py

# Set your actual sampling frequency
SAMPLE_FREQ = fs  # or whatever value was used in Pyroomacoustics

with h5py.File("sim_signals.h5", "r+") as f:
    if 'time_data' in f:
        f['time_data'].attrs['sample_freq'] = SAMPLE_FREQ
        print("Added 'sample_freq' =", SAMPLE_FREQ)
    else:
        print("'/time_data' dataset not found.")

In [ ]:
from pathlib import Path
import acoular as ac
import matplotlib.pyplot as plt

search_freq = 5000
plot_size = int(corner_locs)

# Load mic geometry and signals
ts = ac.TimeSamples(file='sim_signals.h5')
ps = ac.PowerSpectra(source=ts, block_size=512, window='Hanning')

# Define scanning grid around expected source location
rg = ac.RectGrid(x_min=-1, x_max=plot_size, y_min=-1, y_max=plot_size, z=2, increment=0.5)
st = ac.SteeringVector(grid=rg, mics=mg)
bb = ac.BeamformerBase(freq_data=ps, steer=st)

# Beamform at a selected frequency (e.g., 8000 Hz)
pm = bb.synthetic(search_freq, 1) # Frequency of interest, octave range
Lm = ac.L_p(pm)

plt.imshow(Lm.T, origin='lower', vmin=Lm.max()-10, extent=rg.extend(), interpolation='bicubic')
plt.plot(source[0,0], source[1,0], 'x', color='red', markersize=10, label='Actual Source')
plt.plot(mic_locs[0,0], mic_locs[1,0], 'o', color='blue', markersize=10, label='Mic Array 1')
plt.plot(mic_locs[0,pow(GRID_SIDE_LENGTH,2)+1], mic_locs[1,pow(GRID_SIDE_LENGTH,2)+1], 'o', color='blue', markersize=10, label='Mic Array 2')
plt.title("Beamforming Map")
plt.colorbar(label='dB')
plt.legend()
plt.show()


In [ ]:
plt.imshow(Lm.T, origin='lower', vmin=Lm.max()-4, extent=rg.extend(), interpolation='bicubic')
plt.plot(source[0,0], source[1,0], 'x', color='red', markersize=10, label='Actual Source')
plt.plot(mic_locs[0,0], mic_locs[1,0], 'o', color='blue', markersize=10, label='Mic Array 1')
plt.plot(mic_locs[0,pow(GRID_SIDE_LENGTH,2)+1], mic_locs[1,pow(GRID_SIDE_LENGTH,2)+1], 'o', color='blue', markersize=10, label='Mic Array 2')
plt.plot(mic_locs[0,2*pow(GRID_SIDE_LENGTH,2)+1], mic_locs[1,2*pow(GRID_SIDE_LENGTH,2)+1], 'o', color='blue', markersize=10, label='Mic Array 3')
plt.title("Beamforming Map")
plt.colorbar(label='dB')
plt.legend()
plt.show()

## Moving Source


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyroomacoustics as pra
from scipy.io import wavfile
from scipy.signal import fftconvolve  # <--- 1. ADDED THIS IMPORT
from IPython.display import Audio, display

# Room dimensions and parameters
corner_locs = 150.0
h = 9.5
room_dim = [corner_locs, corner_locs, h]
# For outdoor I am using 1.0 as the absorbtion to simulate very little reverb
material = pra.Material(1.0)

# Source and microphone setup
fs, audio = wavfile.read("drone_audio.wav")
if audio.ndim > 1:
    audio = np.mean(audio, axis=1)

# Microphone locations (Assuming append_grid_coordinates and other variables are defined as before)
# ---- Placeholder for your mic setup logic ----
def append_grid_coordinates(start_x, start_y, grid_side_length, mic_spacing, fixed_z):
    all_x_coords, all_y_coords = [], []
    for i in range(grid_side_length):
        for j in range(grid_side_length):
            all_x_coords.append(start_x + i * mic_spacing)
            all_y_coords.append(start_y + j * mic_spacing)
    all_z_coords = [fixed_z] * len(all_x_coords)
    return np.array(all_x_coords), np.array(all_y_coords), np.array(all_z_coords)

grid_side_length = 2
mic_spacing = 0.15
start_x = 1.0
start_y = 1.0
fixed_z = 1.2
all_x_coords, all_y_coords, all_z_coords = append_grid_coordinates(
    start_x=start_x, start_y=start_y, grid_side_length=grid_side_length, mic_spacing=mic_spacing, fixed_z=fixed_z
)
# ---- End placeholder ----

mic_locs = np.array([all_x_coords, all_y_coords, all_z_coords])
mic_array = pra.MicrophoneArray(mic_locs, fs)

# Define the trajectory for the moving source
start_pos = np.array([30.0, 30.0, 4.0])
end_pos = np.array([130.0, 130.0, 4.0])
num_steps = 50
trajectory = np.linspace(start_pos, end_pos, num_steps).T

# Create a Dirac impulse to measure the RIR with ray tracing
# The length determines the max length of the RIR
rir_len_s = 2.0  # seconds
dirac_impulse = np.zeros(int(rir_len_s * fs))
dirac_impulse[0] = 1.0

rirs = []

# Simulate the Room w/ Ray Tracing
print("\nComputing Room Impulse Responses with Ray Tracing (this may take some time)...")
# Define the room corners for the Room object
corners = np.array([[0, 0], [0, room_dim[1]], [room_dim[0], room_dim[1]], [room_dim[0], 0]]).T
for i in range(num_steps):
    # We now use pra.Room, which is more general and supports ray tracing
    room = pra.Room.from_corners(corners, fs=fs, ray_tracing=True, air_absorption=True)
    room.extrude(room_dim[2], materials=material)
    room.add_microphone_array(mic_array)

    # Add a source with the dirac impulse as a signal
    room.add_source(trajectory[:, i], signal=dirac_impulse)

    # Simulat the room with ray tracing
    room.simulate()

    # The RIRs are now in the signals buffer of the microphone array
    rirs.append(room.mic_array.signals)
print("RIR computation complete.")

# Reconstruct audio signal
if not rirs:
    print("Error: No RIRs were computed.")
else:
    max_rir_len = max(rir_set.shape[1] for rir_set in rirs)
    output_len = len(audio) + max_rir_len - 1
    num_mics = mic_array.R.shape[1]
    output_signal = np.zeros((num_mics, output_len))
    audio_segments = np.array_split(audio, num_steps)
    current_pos = 0

    print("\nReconstructing audio signal...")
    for i in range(num_steps):
        segment = audio_segments[i]
        for mic_index in range(num_mics):
            # rirs[i] is now a (num_mics, num_samples) array
            rir = rirs[i][mic_index, :]
            segment_conv = fftconvolve(segment, rir)
            output_signal[mic_index, current_pos : current_pos + len(segment_conv)] += segment_conv
        current_pos += len(segment)
    print("Audio reconstruction complete.")

# --- Visualization (Unchanged) ---
mic_to_listen = 1
print(f"\nSimulated audio of the moving drone (microphone {mic_to_listen + 1}):")
epsilon = np.finfo(float).eps
output_signal_norm = output_signal[mic_to_listen] / (np.max(np.abs(output_signal[mic_to_listen])) + epsilon)
display(Audio(data=output_signal_norm, rate=fs))

print("\nPlotting the waveforms for all microphones:")
fig, ax = plt.subplots(figsize=(15, 8))
time_axis = np.arange(output_signal.shape[1]) / fs
v_offset = 0
for mic_index in range(num_mics):
    signal = output_signal[mic_index]
    signal_norm = signal / (np.max(np.abs(signal)) + epsilon)
    ax.plot(time_axis, signal_norm + v_offset, label=f'Mic {mic_index + 1}')
    v_offset += 2
ax.set_title('Simulated Signals for All Microphones')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Normalized Amplitude (Vertically Offset)')
ax.set_yticks([])
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(subplot_kw={'projection': '3d'})
ax.plot(mic_locs[0, :], mic_locs[1, :], mic_locs[2, :], 'o', label='Microphones')
ax.plot(trajectory[0, :], trajectory[1, :], trajectory[2, :], 'x-', label='Drone Trajectory')
ax.set_xlabel("X-coordinate (m)")
ax.set_ylabel("Y-coordinate (m)")
ax.set_zlabel("Z-coordinate (m)")
ax.set_title("Drone Trajectory and Microphone Array")
ax.legend()
plt.grid()
plt.show()

In [ ]:
import graphviz

# Create a new directed graph
dot = graphviz.Digraph('SpatialAudioFlowchart', comment='Flowchart from Pyroomacoustics to Acoular')
dot.attr(rankdir='TD', label='Flowchart: Microphone Array -> Pyroomacoustics -> Acoular', fontsize='20')
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue', fontname='Helvetica')
dot.attr('edge', fontname='Helvetica')

# 1. Define Acoustic Scene Stage (using a subgraph for grouping)
with dot.subgraph(name='cluster_0') as c:
    c.attr(label='1. Define Acoustic Scene', style='filled', color='lightgrey')
    c.node('A', 'Microphone Array & Sound Source Locations')

# 2. Simulate the Acoustic Environment Stage
with dot.subgraph(name='cluster_1') as c:
    c.attr(label='2. Simulate the Acoustic Environment', style='filled', color='lightgrey')
    c.node('B', 'Pyroomacoustics\nSpatial Audio Simulation')
    c.node('C', 'Simulated Microphone Signals\n(Multi-channel Audio)')

# 3. Analyze the Simulated Data Stage
with dot.subgraph(name='cluster_2') as c:
    c.attr(label='3. Analyze the Simulated Data', style='filled', color='lightgrey')
    c.node('D', 'Acoular\nBeamforming and Localization')
    c.node('E', 'Output: Sound Source Map')

# Define the flow of the chart by connecting the nodes
dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('C', 'D')
dot.edge('D', 'E')

# Specify a different style for the final output node
dot.node('E', style='rounded,filled', fillcolor='lightgreen')


# Render the graph to a file (e.g., PNG, PDF, SVG)
# The file will be saved in the same directory where the script is run.
output_filename = 'spatial_audio_flowchart'
dot.render(output_filename, format='png', view=False, cleanup=True)

print(f"Flowchart saved as '{output_filename}.png'")